In [ ]:
import cv2 as cv
import numpy as np
import matplotlib.pyplot as plt
import glob


# Checkerboard sizes, the first value is the number of checkerboards of specified size, the second is the vertical size and the last is the horizontal size
checkerboard_sizes = [(1,15,5), (2,7,11), (5,5,7), (5,7,5)]

images_left = glob.glob('calib/image_02/data/*.png')
images_right = glob.glob('calib/image_03/data/*.png')



images_left = images_left[:5]
images_right = images_right[:5]

print(len(images_left), len(images_right))

# From [[0, -0.25, 0], [-0.25, 2, -0.25], [0, -0.25, 0]]
#sharpening_kernel_count = 10
# From (3,3), ...
#gauss_sizes_count = 3

square_size = 10

def search_for_optimal_kernel(images, sharpening_kernel_count, filter_size_max):
    corners_for_all_pics = []
    object_points_for_all_pics = []
    starting_sharpening_kernel = 5
    starting_filter_size = 0
    starting_clip_Limit = 1
    
    # Loop for all the images
    for name in images:
        
        checkerboard_sizes_copy = np.array(checkerboard_sizes)
        checkerboard_points = []
        checkerboards_detected = 0
        allCheckerboard_detected = False
        
        # Loop for all the different sizes in checkerboard_sizes
        for size_number in range(len(checkerboard_sizes_copy)):
            
            # Loop for all the instances of the same checkerboard size
            for c in range(checkerboard_sizes_copy[size_number][0]):
                
                size = (checkerboard_sizes_copy[size_number][1], checkerboard_sizes_copy[size_number][2])
                
                chessboard_coords = np.zeros((size[0] * size[1], 3), np.float32)
                chessboard_coords[:, :2] = np.mgrid[0:size[0], 0:size[1]].T.reshape(-1, 2)
                chessboard_coords *= square_size
                
                allSizes_detected = False
                
                # Loop for different values of sharpening kernel
                for i in range(starting_sharpening_kernel, sharpening_kernel_count):
                    
                    
                    # Calculate sharpening kernel
                    sides = -0.25*(i+1) + 0.25
                    sharpening_kernel = np.array([[0, sides, 0], [sides, i+1, sides], [0, sides, 0]])
                    
                    
                    # Loop for different values of gauss blur filter
                    for j in range(starting_filter_size, filter_size_max):
                        filter_size = j * 2 + 1

                        
                        # Loop for different clip limit values for CLAHE
                        for clip_Limit in range(starting_clip_Limit, 4):
                            
                            
                        
                            img = cv.imread(name)
                            gray = cv.cvtColor(img, cv.COLOR_BGR2GRAY)
                            
                        
                            clahe = cv.createCLAHE(clipLimit=clip_Limit*2, tileGridSize=(16, 16))
                            gray = clahe.apply(gray)
                            gray = cv.GaussianBlur(gray, (filter_size, filter_size), 0)
                            gray = cv.filter2D(gray, -1, sharpening_kernel)
                            
                            
                            
                            # Masking the checkerboards that were detected
                            for checkerboard in checkerboard_points:
                                for p in range(len(checkerboard)):
                                    if p < len(checkerboard) -1:
                                        cv.line(gray, checkerboard[p], checkerboard[p+1], color = (0,0,0), thickness=20)
                                    else:
                                        cv.line(gray, checkerboard[0], checkerboard[p], color = (0,0,0), thickness=20)

                                cv.fillConvexPoly(gray, checkerboard, color=(0,0,0))
                            
                            cv.imshow('img',gray)
                            cv.waitKey(1)
                            
                            
                                        

                            ret, corners = cv.findChessboardCorners(gray, (size), cv.CALIB_CB_ADAPTIVE_THRESH | cv.CALIB_CB_NORMALIZE_IMAGE)
                                            
                            if ret:
                                gray = cv.drawChessboardCorners(gray, (size), corners,ret)
                                            
                                checkerboard_point = np.array([corners[0][0], corners[size[0]-1][0], corners[-1][0], corners[size[0]*size[1]-size[0]][0]], dtype=np.int32)
                                checkerboard_points.append(checkerboard_point)
        
                                object_points_for_all_pics.append(chessboard_coords)    
                                corners_for_all_pics.append(corners)
                                
                                # Masking the checkerboard
                                for p in range(len(checkerboard_point)):
                                    if p < len(checkerboard_point) -1:
                                        cv.line(gray, checkerboard_point[p], checkerboard_point[p+1], color = (0,0,0), thickness=20)
                                    else:
                                        cv.line(gray, checkerboard_point[0], checkerboard_point[p], color = (0,0,0), thickness=20)

                                cv.fillConvexPoly(gray, checkerboard_point, color=(0,0,0))

                                        
                                checkerboards_detected += 1
                                checkerboard_sizes_copy[size_number][0] -= 1
                                
                                cv.imshow('img',gray)
                                cv.waitKey(1)
                                
                                print("Checkerboard: ", checkerboard_sizes_copy[size_number],  ", Detected for params: ",i, j, clip_Limit)
                                
                                if checkerboards_detected == 13:
                                    allCheckerboard_detected = True
                                    break
                                
                                if checkerboard_sizes_copy[size_number][0] == 0:
                                    allSizes_detected = True
                                    break
                            
                                
                            
                        if allCheckerboard_detected or allSizes_detected:
                            break
                    if allCheckerboard_detected or allSizes_detected:
                        break    
                                        
                                        
                if allCheckerboard_detected or allSizes_detected:
                    break
                else:
                    print("Couldn't detect checkerboard of size", checkerboard_sizes_copy[size_number])
                        
                    
            if allCheckerboard_detected:
                print("All checkerboards detected image: ", name)
                break 
        
                           
    # Calculate camera parameters
    print("Calibrating the camera....")
    ret, camera_matrix, dist_coeffs, rot_vecs, trans_vecs = cv.calibrateCamera(object_points_for_all_pics, corners_for_all_pics, gray.shape[::-1], None, None)                
            
            
    
    
    return camera_matrix, dist_coeffs, rot_vecs, trans_vecs, object_points_for_all_pics, corners_for_all_pics





camera_matrix_left, dist_coeffs_left, rot_vecs_left, trans_vecs_left, object_points_left, corners_for_left = search_for_optimal_kernel(images_left, 12, 3)

camera_matrix_right, dist_coeffs_right, rot_vecs_right, trans_vecs_right, object_points_right, corners_for_right = search_for_optimal_kernel(images_right, 12, 3)






demo_img = cv.imread("seq_01/image_02/data/000000.png")
height, width, _ = demo_img.shape
print("Calculating new camera matrix.....")
new_camera_matrix_left, roi = cv.getOptimalNewCameraMatrix(camera_matrix_left, dist_coeffs_left, (width, height), 1, (width, height))
new_camera_matrix_right, roi = cv.getOptimalNewCameraMatrix(camera_matrix_right, dist_coeffs_right, (width, height), 1, (width, height))
print("Undistorting the image")
# ==================== Undistort the image ====================
#undistorted_demo_img = cv.undistort(demo_img, camera_matrix, dist_coeffs, None, new_camera_matrix)
        
# ==================== Crop the image to the valid region of interest ====================
# x, y, w, h = roi
# undistorted_demo_img = undistorted_demo_img[y:y+h, x:x+w]


# fig, ax = plt.subplots(nrows=2, ncols=1, figsize=(15,15))
# ax[0].imshow(demo_img)
# ax[0].set_title('Original image')
# ax[1].imshow(undistorted_demo_img)
# ax[1].set_title('Undistorted image')


# 




cv.destroyAllWindows()

1 1
Checkerboard:  [ 0 15  5] , Detected for params:  5 1 1
Checkerboard:  [ 1  7 11] , Detected for params:  5 0 1
Checkerboard:  [ 0  7 11] , Detected for params:  5 2 1
Checkerboard:  [4 5 7] , Detected for params:  5 0 1
Checkerboard:  [3 5 7] , Detected for params:  5 0 2
Checkerboard:  [2 5 7] , Detected for params:  5 0 3
Checkerboard:  [1 5 7] , Detected for params:  5 1 1
Checkerboard:  [0 5 7] , Detected for params:  5 1 2
Checkerboard:  [4 7 5] , Detected for params:  5 0 1
Checkerboard:  [3 7 5] , Detected for params:  5 0 2
Checkerboard:  [2 7 5] , Detected for params:  5 0 3
Checkerboard:  [1 7 5] , Detected for params:  5 1 1
Checkerboard:  [0 7 5] , Detected for params:  5 1 2
All checkerboards detected image:  calib/image_02/data\0000000000.png
Calibrating the camera....
Checkerboard:  [ 0 15  5] , Detected for params:  7 1 2
Checkerboard:  [ 1  7 11] , Detected for params:  5 0 1
Checkerboard:  [ 0  7 11] , Detected for params:  5 2 1
Checkerboard:  [4 5 7] , Detected

In [13]:


ret, cameraMatrix1, distCoeffs1, cameraMatrix2, distCoeffs2, R, T, E, F = cv.stereoCalibrate(
    object_points_right,     # 3D points in real-world space
    corners_for_left,     # 2D points in left camera
    corners_for_right,     # 2D points in right camera
    camera_matrix_left,    # Intrinsic matrix of the left camera
    dist_coeffs_left,      # Distortion coefficients of the left camera
    camera_matrix_right,    # Intrinsic matrix of the right camera
    dist_coeffs_right,      # Distortion coefficients of the right camera
    (width, height),        # Image resolution (e.g., (width, height))
    flags=cv.CALIB_FIX_INTRINSIC
)


# Rectification for stereo cameras
R_left_rect, R_right_rect, P_left, P_right, Q, validPixROI1, validPixROI2 = cv.stereoRectify(
    cameraMatrix1, distCoeffs1,
    cameraMatrix2, distCoeffs2,
    (width, height),  # Image resolution
    R,          # Rotation matrix from stereoCalibrate
    T,          # Translation vector from stereoCalibrate
    flags=cv.CALIB_ZERO_DISPARITY,  # Assume no disparity shift
    alpha=0     # Free scaling (0 means crop, 1 means no crop)
)





In [14]:
R_left, _ = cv.Rodrigues(rot_vecs_left[0])
R_right, _ = cv.Rodrigues(rot_vecs_right[0])

R_left = np.round(R_left,5)
R_right = np.round(R_right,5)

T_left = np.round(trans_vecs_left,3)
T_right = np.round(trans_vecs_right,3)


x, y, w, h = roi
S_rect_left = (w, h)
S_rect_right = (w, h)


print("Shape: ", (width,height))
print("Camera matrix left: ",camera_matrix_left, "\n=======================\n", "Camera matrix right",camera_matrix_right, "\n")
print("=======================")
print("Dist coeffs left: ",dist_coeffs_left, "\n=======================\n","Dist coeffs right", dist_coeffs_right, "\n")
print("=======================")
print("R left: ",R_left, "\n=======================\n","R right: ", R_right, "\n")
print("=======================")
print("T left: ",T_left.flatten()[:3]/1000, "\n=======================\n","T right: ", T_right.flatten()[:3]/1000, "\n")
#print("=======================")
#print("S rect left: ", S_rect_left, "\n=======================\n", "S rect right: ",S_rect_right, "\n")
print("=======================")
print("R rect left: ", R_left_rect, "\n=======================\n","R rect left: ", R_right_rect, "\n")
print("=======================")
print("P left: ",P_left, "\n=======================\n","P right: ", P_right, "\n")

Shape:  (1224, 370)
Camera matrix left:  [[1.14223274e+03 0.00000000e+00 6.91012626e+02]
 [0.00000000e+00 1.04598523e+03 2.55291335e+02]
 [0.00000000e+00 0.00000000e+00 1.00000000e+00]] 
 Camera matrix right [[909.83553072   0.         681.63548497]
 [  0.         918.0381791  245.91958196]
 [  0.           0.           1.        ]] 

Dist coeffs left:  [[-2.14331759e-01 -3.16822763e-01  5.36303124e-05 -1.24748973e-02
   4.95565159e-01]] 
 Dist coeffs right [[-4.40872882e-01  3.45218442e-01  1.14047927e-04  1.68198599e-03
  -1.60444568e-01]] 

R left:  [[-0.09101  0.04771  0.99471]
 [ 0.99473  0.05168  0.08854]
 [-0.04718  0.99752 -0.05216]] 
 R right:  [[-0.08283  0.09571  0.99196]
 [ 0.99509  0.06201  0.07711]
 [-0.05413  0.99348 -0.10038]] 

T left:  [ 0.349806 -0.041265  0.577735] 
 T right:  [ 0.321391 -0.042839  0.508331] 

R rect left:  [[ 0.97815008 -0.20752706 -0.01244737]
 [ 0.20727531  0.97809935 -0.01893674]
 [ 0.01610465  0.01594294  0.9997432 ]] 
 R rect left:  [[ 0.81873